In [ ]:
import yaml
from app.datasets.loader import load_multiple_test_cases, load_test_cases
from app.datasets.validator import validate_dataset_schema
from app.datasets.validator import validate_dataset_schema
from app.client.rag_client import RAGClient
from app.tests.run_tests import run_tests
from app.tests.nodes.reformulate import send_reformulate_requests, save_reformualte_responses, reformulate_tests

In [ ]:
file_list = [
  './app/data/raw/rapido.xlsx',
]

test_config = {
  'GENERAL_TESTS': True,
  'TIMINGS': {'test': True, 'report': False},
  'TOKENS': {'test': True, 'report': False},
  'FOUNDRYS': {'test': True, 'report': False},
  'TRIAGE': {'test': False, 'report': False},
  'ROUTER': {'test': False, 'report': False},
  'GROUNDING': {'test': False, 'report': False},
  'SAVE_RESULTS': False,
  'PATH': './app/data/processed/reports/report_RAPIDO',
  
  'REFORMULATE': {'test': False, 'report': False}
}   

if file_list: 
  df = load_multiple_test_cases(file_list)
  df = validate_dataset_schema(df)

with open('./app/config/config.yaml', 'r') as file:
  config_data = yaml.load(file, Loader= yaml.FullLoader) 
  
client = RAGClient(config_data)
test_timestamps = {}

In [ ]:
# TEST GENERALES
if test_config.get('GENERAL_TESTS', False):
  responses = client.query_batch(df['user_input'],df['reference'])
  save_responses_in_json, response_file_path = client.save_api_responses(responses)
  test_timestamps['general_tests'] = str(response_file_path).replace('\\', '/')

In [ ]:
import json

response_file_path = './app/data/processed/outcomes/outcome_20260507-130815.json'
with open(response_file_path, 'r', encoding='UTF-8') as f:
  responses = json.load(f)
test_timestamps['general_tests'] = 'outcome_20260507-130815.json'

In [47]:
from collections import Counter
from itertools import chain

def foundrys_tests(data: list[dict]) -> dict:
  links = []
  result = {}
  for item in data:
    
    if item is None or 'ok' in item:
      continue
    
    if 'router' in item['partial_answers'] and 'grounding' in item['partial_answers']:
      node_metadata = item['node_metadata']
      route = item['partial_answers'].get('router', {'route': ''}).get('route', '')
      
      ref_link = node_metadata.get('reformulate').get('endpoint_routing')[0].get('endpoint').split(':')[0]
      tri_link = node_metadata.get('triage').get('endpoint_routing')[0].get('endpoint', '').split(':')[0]
    
      rou_link = '99'

      if node_metadata['semantic_router']['endpoint_routing'][0].get('endpoint'):
        rou_link = node_metadata['semantic_router']['endpoint_routing'][0].get('endpoint').split(':')[0]
      else:
        node_metadata['llm_router']['endpoint_routing'][0].get('endpoint').split(':')[0]


      per_link = node_metadata.get('personality', {'endpoint': 'x'}).get('endpoint_routing')[0].get('endpoint', 'x').split(':')[0]
      gro_link = node_metadata.get('grounding', {'endpoint': 'x'}).get('endpoint_routing')[0].get('endpoint', 'x').split(':')[0]
      
      ree_link = node_metadata.get(f'agent_{route}', {'retrieve_embeddings', 'x'}).get('retrieve_embeddings','x').get('endpoint_routing')[0].get('endpoint', 'x').split(':')[0]
      rag_link = node_metadata.get(f'agent_{route}', {'rag_answer', 'x'}).get('rag_answer','x').get('endpoint_routing')[0].get('endpoint', 'x').split(':')[0]

      links.append([
        ref_link, 
        tri_link, 
        rou_link, 
        per_link, 
        gro_link, 
        ree_link, 
        rag_link
      ])

  total_counts = Counter(chain.from_iterable(links))
  del total_counts['x']
  
  total = total_counts.total()
  result['total'] = total
  
  for i in range(len(total_counts)):    
    result[f'{i}'] = {
      'count': total_counts[f'{i}'],
      'percentage': round(total_counts[f'{i}'] * 100 / total, 2)
    }
  
  return result

In [48]:
foundrys_results = foundrys_tests(responses)

In [49]:
print(foundrys_results)

{'total': 574, '0': {'count': 63, 'percentage': 10.98}, '1': {'count': 60, 'percentage': 10.45}, '2': {'count': 58, 'percentage': 10.1}, '3': {'count': 56, 'percentage': 9.76}, '4': {'count': 55, 'percentage': 9.58}, '5': {'count': 56, 'percentage': 9.76}, '6': {'count': 58, 'percentage': 10.1}, '7': {'count': 57, 'percentage': 9.93}, '8': {'count': 53, 'percentage': 9.23}, '9': {'count': 58, 'percentage': 10.1}}


In [ ]:
if test_config.get('GENERAL_TESTS'):
  results, reports = run_tests(
    config = test_config, 
    data = responses, 
    df = df, 
    timestamp = test_timestamps
)

In [ ]:
print(results)